In [0]:
# ============================================================
# PROJETO: UCI Online Retail
# CAMADA: SILVER
# ============================================================
#
# Objetivo:
# Limpar, padronizar e enriquecer os dados provenientes
# da camada Bronze.
#
# Princípios:
# 1. Preservar a rastreabilidade dos dados.
# 2. Não alterar a tabela Bronze original.
# 3. Documentar todas as transformações.
# 4. Registrar regras de negócio nos metadados.
# 5. Preparar os dados para análises e agregações na Gold.
# 6. Fornecer contexto estruturado para consumo futuro
#    por ferramentas analíticas e soluções de IA.
#
# Fonte:
# UCI Online Retail Dataset
#
# Camada de origem:
# Bronze — dados próximos à fonte original.
#
# Camada de destino:
# Silver — dados limpos, padronizados e enriquecidos.
# ============================================================

In [0]:
# ============================================================
# METADADOS — Carga da camada Bronze
# ============================================================
# Origem:
# Tabela Delta localizada na camada Bronze.
#
# Objetivo:
# Utilizar os dados originais armazenados na Bronze como
# fonte para as transformações da camada Silver.
#
# Regra:
# A tabela Bronze não será modificada diretamente.
# Todas as transformações serão realizadas em um novo
# DataFrame destinado à camada Silver.
# ============================================================

silver_df = spark.table("uci_retail.bronze.online_retail")

In [0]:
silver_df.count()

In [0]:
display(silver_df.limit(10))

In [0]:
# ============================================================
# METADADOS — TransactionType
# ============================================================
# Nome: TransactionType
# Tipo: STRING
# Camada: Silver
# Origem: InvoiceNo
#
# Descrição:
# Classifica cada registro de acordo com o tipo de transação.
#
# Regra de negócio:
# - InvoiceNo iniciado por "C" = Cancellation
# - Demais registros = Sale
#
# Objetivo:
# Permitir a diferenciação entre vendas e cancelamentos
# sem excluir os registros de cancelamento.
#
# Rastreabilidade:
# InvoiceNo original permanece preservado.
#
# Observação:
# Foram identificados anteriormente 9.288 registros cujo
# InvoiceNo começa com "C".
# ============================================================

from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "TransactionType",
    when(col("InvoiceNo").startswith("C"), "Cancellation")
    .otherwise("Sale")
)

In [0]:
display(
    silver_df.groupBy("TransactionType").count()
)

In [0]:
silver_df.printSchema()

In [0]:
from pyspark.sql.functions import col, sum

display(
    silver_df.select(
        [
            sum(col(c).isNull().cast("int")).alias(c)
            for c in silver_df.columns
        ]
    )
)

In [0]:
# ============================================================
# METADADOS — CustomerID
# ============================================================
# Nome: CustomerID
# Tipo original: DOUBLE
# Tipo planejado na Silver: STRING
# Origem: Dataset UCI Online Retail
# Camada: Silver
#
# Descrição:
# Identificador único associado ao cliente responsável pela
# transação.
#
# Qualidade dos dados:
# Foram identificados 135.080 registros sem CustomerID.
#
# Tratamento:
# Os valores NULL serão preservados.
#
# Justificativa:
# A ausência de CustomerID não significa que a transação
# seja inválida. A venda pode possuir informações válidas
# de produto, quantidade, preço e data mesmo sem identificação
# do cliente.
#
# Regra para IA:
# NULL em CustomerID significa "cliente não identificado /
# informação de cliente não disponível".
# NULL NÃO deve ser interpretado como CustomerID = 0.
#
# Impacto analítico:
# Métricas baseadas em clientes identificados devem considerar
# somente registros que possuam CustomerID.
# Métricas gerais de vendas podem utilizar registros sem
# CustomerID, desde que a regra da métrica permita.
# ============================================================

In [0]:
display(
    silver_df
    .filter(
        col("CustomerID").isNotNull() &
        (col("CustomerID") != col("CustomerID").cast("long"))
    )
    .select("CustomerID")
    .limit(20)
)

In [0]:
# ============================================================
# METADADOS — Padronização do CustomerID
# ============================================================
# Campo: CustomerID
#
# Tipo original:
# DOUBLE
#
# Tipo Silver:
# STRING
#
# Motivo:
# CustomerID é um identificador de cliente e não uma medida
# numérica. Portanto, não deve ser utilizado em operações
# matemáticas.
#
# Validação:
# Foi verificado que os CustomerID não possuem valores
# decimais reais entre os registros não nulos.
#
# Tratamento de NULL:
# Os 135.080 valores NULL serão preservados.
#
# Regra para IA:
# CustomerID deve ser interpretado como identificador.
# Não realizar soma, média ou outras operações matemáticas
# sobre esse campo.
# ============================================================

In [0]:
silver_df = silver_df.withColumn(
    "CustomerID",
    col("CustomerID").cast("long").cast("string")
)

In [0]:
silver_df.printSchema()

In [0]:
silver_df.filter(col("CustomerID").isNull()).count()

In [0]:
display(
    silver_df
    .select("Quantity")
    .summary("count", "min", "max", "mean", "stddev")
)

In [0]:
display(
    silver_df
    .filter(col("Quantity") < 0)
    .select("InvoiceNo", "StockCode", "Quantity", "UnitPrice", "TransactionType")
    .limit(20)
)

In [0]:
# ============================================================
# METADADOS — Quantity
# ============================================================
# Campo: Quantity
# Tipo: LONG
# Origem: Dataset UCI Online Retail
# Camada: Silver
#
# Descrição:
# Quantidade de unidades registrada na linha da transação.
#
# Validação:
# Valores negativos serão investigados antes de qualquer
# remoção ou transformação.
#
# Regra de negócio:
# A interpretação de valores negativos deve considerar o
# TransactionType e o contexto da transação.
#
# Regra para IA:
# Quantity representa quantidade de unidades e não deve ser
# interpretada automaticamente como valor monetário.
# ============================================================

In [0]:
display(
    silver_df
    .select("Quantity")
    .summary("count", "min", "max", "mean", "stddev")
)

In [0]:
display(
    silver_df
    .filter(col("Quantity") < 0)
    .select(
        "InvoiceNo",
        "StockCode",
        "Quantity",
        "UnitPrice",
        "TransactionType"
    )
    .limit(20)
)

In [0]:
display(
    silver_df
    .select("UnitPrice")
    .summary("count", "min", "max", "mean", "stddev")
)

In [0]:
display(
    silver_df
    .filter(col("UnitPrice") <= 0)
    .select(
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "TransactionType"
    )
    .limit(30)
)

In [0]:
from pyspark.sql.functions import when, col, count, sum

display(
    silver_df
    .groupBy("TransactionType")
    .agg(
        count("*").alias("TotalRecords"),
        sum(when(col("UnitPrice") == 0, 1).otherwise(0)).alias("ZeroUnitPrice"),
        sum(when(col("UnitPrice") < 0, 1).otherwise(0)).alias("NegativeUnitPrice"),
        sum(when(col("Quantity") < 0, 1).otherwise(0)).alias("NegativeQuantity")
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "DataQualityStatus",
    when(
        (col("Quantity") <= 0) | (col("UnitPrice") <= 0),
        "Review"
    ).otherwise("Valid")
)

In [0]:
display(
    silver_df
    .groupBy("DataQualityStatus")
    .count()
)

In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.withColumn(
    "Revenue",
    col("Quantity") * col("UnitPrice")
)

In [0]:
display(
    silver_df
    .select(
        "InvoiceNo",
        "StockCode",
        "Quantity",
        "UnitPrice",
        "TransactionType",
        "DataQualityStatus",
        "Revenue"
    )
    .limit(20)
)

In [0]:
display(
    silver_df
    .select("Revenue")
    .summary("count", "min", "max", "mean", "stddev")
)

In [0]:
display(
    silver_df
    .select(
        "TransactionType",
        "DataQualityStatus",
        "Revenue"
    )
    .groupBy(
        "TransactionType",
        "DataQualityStatus"
    )
    .agg(
        count("*").alias("TotalRecords"),
        sum("Revenue").alias("TotalRevenue")
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "DataQualityStatus",
    when(
        col("TransactionType") == "Cancellation",
        "Valid"
    ).when(
        (col("Quantity") <= 0) | (col("UnitPrice") <= 0),
        "Review"
    ).otherwise("Valid")
)

In [0]:
display(
    silver_df
    .groupBy("TransactionType", "DataQualityStatus")
    .count()
)

In [0]:
# ============================================================
# METADADOS — Revenue
# ============================================================
# Nome: Revenue
# Tipo: DOUBLE
# Camada: Silver
# Origem: Quantity e UnitPrice
#
# Descrição:
# Representa o impacto financeiro de cada linha da transação.
#
# Regra:
# Revenue = Quantity * UnitPrice
#
# Interpretação:
# - Valor positivo: impacto financeiro de uma venda.
# - Valor negativo: impacto financeiro de um cancelamento.
# - Valor zero: linha sem impacto financeiro.
#
# Observação:
# Cancelamentos não são removidos da Silver. Quando a quantidade
# é negativa, o Revenue também representa um valor negativo,
# permitindo analisar posteriormente o impacto dos cancelamentos.
#
# Uso futuro:
# A coluna será utilizada na camada Gold para criação de métricas
# como receita, vendas por período, país e produto.
# ============================================================

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uci_retail.silver.online_retail")

In [0]:
silver_check = spark.table("uci_retail.silver.online_retail")

print("Quantidade de registros:", silver_check.count())

silver_check.printSchema()

In [0]:
display(
    silver_check
    .select(
        "InvoiceNo",
        "StockCode",
        "Quantity",
        "UnitPrice",
        "CustomerID",
        "TransactionType",
        "DataQualityStatus",
        "Revenue"
    )
    .limit(20)
)

In [0]:
# Pipeline de Engenharia de Dados - UCI Online Retail